# Neural Burst Simulation

Interactive version of the neural burst simulation and detection pipeline.

All analysis parameters are exposed below and can be changed directly in the notebook.

The original modular command-line pipeline remains available with:

`python run_pipeline.py`

## 1. Setup

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd

from neurodsp.sim import sim_combined
from neurodsp.utils import create_times, set_random_seed
from neurodsp.burst import detect_bursts_dual_threshold, compute_burst_stats
from neurodsp.plts.time_series import plot_time_series, plot_bursts

cwd = Path.cwd()

if (cwd / "run_pipeline.py").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "run_pipeline.py").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root")

os.chdir(PROJECT_ROOT)

Path("data").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)

## 2. Parameters

Edit the values in this cell and rerun the cells below.

The default values reproduce the original Python pipeline.

In [ ]:
# Reproducibility
SEED = 42

# General simulation
FS = 1000
N_SECONDS = 10

# Aperiodic / synaptic-current component
N_NEURONS = 1000
FIRING_RATE = 2
T_KER = 1.0
TAU_R = 0.002
TAU_D = 0.02

# Bursty oscillation
BETA_FREQUENCY = 20
ENTER_BURST = 0.2
LEAVE_BURST = 0.2

# Burst detection
BETA_RANGE = (13, 30)
DUAL_THRESHOLD = (1, 2)

print("Sampling frequency:", FS, "Hz")
print("Duration:", N_SECONDS, "s")
print("Random seed:", SEED)
print("Beta frequency:", BETA_FREQUENCY, "Hz")
print("Beta detection range:", BETA_RANGE, "Hz")
print("Dual threshold:", DUAL_THRESHOLD)

## 3. Simulate neural activity

The signal combines an aperiodic synaptic-current component with transient beta oscillations.

In [ ]:
set_random_seed(SEED)

components = {
    "sim_synaptic_current": {
        "n_neurons": N_NEURONS,
        "firing_rate": FIRING_RATE,
        "t_ker": T_KER,
        "tau_r": TAU_R,
        "tau_d": TAU_D,
    },
    "sim_bursty_oscillation": {
        "freq": BETA_FREQUENCY,
        "enter_burst": ENTER_BURST,
        "leave_burst": LEAVE_BURST,
    },
}

signal = sim_combined(
    n_seconds=N_SECONDS,
    fs=FS,
    components=components,
)

times = create_times(
    n_seconds=N_SECONDS,
    fs=FS,
)

simulated_data = pd.DataFrame({
    "time_s": times,
    "signal": signal,
})

simulated_data.to_csv(
    "data/simulated_signal.csv",
    index=False,
)

print("Saved simulated signal to data/simulated_signal.csv")
print("Number of samples:", len(signal))

## 4. Plot simulated signal

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

plot_time_series(
    times,
    signal,
    ax=ax,
)

ax.set_title("Simulated neural signal")
fig.tight_layout()

fig.savefig(
    "figures/raw_signal.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

## 5. Detect beta bursts

In [ ]:
bursting = detect_bursts_dual_threshold(
    signal,
    FS,
    DUAL_THRESHOLD,
    BETA_RANGE,
)

burst_stats = compute_burst_stats(
    bursting,
    FS,
)

burst_detection = pd.DataFrame({
    "time_s": times,
    "signal": signal,
    "detected_burst": bursting,
})

burst_detection.to_csv(
    "results/burst_detection.csv",
    index=False,
)

pd.DataFrame([burst_stats]).to_csv(
    "results/burst_statistics.csv",
    index=False,
)

print("Burst detection complete:")

for name, value in burst_stats.items():
    print(f"{name}: {value}")

## 6. Burst statistics

In [ ]:
stats = pd.DataFrame([burst_stats])
stats

## 7. Visualize detected bursts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

plot_bursts(
    times,
    signal,
    bursting,
    ax=ax,
)

ax.set_title("Detected beta bursts")
fig.tight_layout()

fig.savefig(
    "figures/burst_detection.png",
    dpi=150,
    bbox_inches="tight",
)

plt.show()

## 8. Interactive exploration

Change any parameter in the **Parameters** cell and rerun the simulation, burst-detection, statistics, and plotting cells.

Examples include:

- `SEED`
- `FS`
- `N_SECONDS`
- `N_NEURONS`
- `FIRING_RATE`
- `T_KER`
- `TAU_R`
- `TAU_D`
- `BETA_FREQUENCY`
- `ENTER_BURST`
- `LEAVE_BURST`
- `BETA_RANGE`
- `DUAL_THRESHOLD`

The default settings correspond to the script-based analysis.

## Reproducibility

The notebook is intended for interactive exploration in Binder.

The modular Python scripts remain the canonical reproducible pipeline:

`python run_pipeline.py`